In [0]:
# ============================================================
# UK RETAIL COMMERCIAL INTELLIGENCE PLATFORM
# MODULE 1 — SILVER CLEANING & COMMERCIAL P&L
# Author: Byron Kamau
# ============================================================
# DEPENDS ON: Module 7 (all bronze + silver foundation tables)
#
# OUTPUTS:
#   silver_pos_cleaned         — clean, typed, enriched POS data
#   gold_retailer_pl           — retailer-level P&L waterfall
#   gold_sku_profitability     — SKU margin ranking with flags
#   gold_regional_performance  — sales & margin by UK region
#   gold_margin_waterfall      — margin bridge data for viz
# ============================================================

# Module 1 — Silver Cleaning & Commercial P&L
**Layer:** Bronze → Silver → Gold

**What this notebook does:**
1. Cleans and types the raw POS transactions (Bronze → Silver)
2. Enriches with ERP product master, retailer funding, UK region
3. Builds the full commercial P&L waterfall (Gross → Net → Margin)
4. Produces SKU profitability ranking
5. Produces regional performance table
6. Produces margin waterfall bridge for Power BI

## Cell 1 — Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

DATABASE = "uk_retail_commercial"
spark.sql(f"USE {DATABASE}")

print("✅ Database set:", DATABASE)
print("📋 Available tables:")
spark.sql(f"SHOW TABLES IN {DATABASE}").show(20, truncate=False)

✅ Database set: uk_retail_commercial
📋 Available tables:
+--------------------+--------------------------------+-----------+
|database            |tableName                       |isTemporary|
+--------------------+--------------------------------+-----------+
|uk_retail_commercial|bronze_ons_retail_index         |false      |
|uk_retail_commercial|bronze_ons_retail_timeseries    |false      |
|uk_retail_commercial|bronze_pos_raw                  |false      |
|uk_retail_commercial|bronze_trends_baby_regions      |false      |
|uk_retail_commercial|bronze_trends_baby_timeseries   |false      |
|uk_retail_commercial|bronze_trends_nursery_regions   |false      |
|uk_retail_commercial|bronze_trends_nursery_timeseries|false      |
|uk_retail_commercial|gold_competitor_pricing         |false      |
|uk_retail_commercial|gold_forecast_accuracy          |false      |
|uk_retail_commercial|gold_forecast_scenarios         |false      |
|uk_retail_commercial|gold_margin_waterfall           |fals

## Cell 2 — Preview Raw POS Data
Understand what we're working with before cleaning

In [0]:
df_raw = spark.table(f"{DATABASE}.bronze_pos_raw")

print(f"Total raw rows : {df_raw.count():,}")
print(f"Columns        : {df_raw.columns}")
print()

# Check data types as ingested
df_raw.printSchema()

# Preview
display(df_raw.limit(10))

Total raw rows : 1,067,371
Columns        : ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'CustomerID', 'Country', 'source_sheet', 'ingested_at']

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- source_sheet: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,source_sheet,ingested_at
502059,22488,NATURAL SLATE RECTANGLE CHALKBOARD,12,2010-03-22 14:44:00,1.65,16560,United Kingdom,2009-2010,2026-03-18T10:29:47.580Z
502059,22360,GLASS JAR ENGLISH CONFECTIONERY,12,2010-03-22 14:44:00,2.95,16560,United Kingdom,2009-2010,2026-03-18T10:29:47.580Z
502059,22362,GLASS JAR PEACOCK BATH SALTS,6,2010-03-22 14:44:00,2.95,16560,United Kingdom,2009-2010,2026-03-18T10:29:47.580Z
502059,22363,GLASS JAR MARMALADE,6,2010-03-22 14:44:00,2.95,16560,United Kingdom,2009-2010,2026-03-18T10:29:47.580Z
502059,22364,GLASS JAR DIGESTIVE BISCUITS,6,2010-03-22 14:44:00,2.95,16560,United Kingdom,2009-2010,2026-03-18T10:29:47.580Z
502059,22359,GLASS JAR KINGS CHOICE,6,2010-03-22 14:44:00,2.95,16560,United Kingdom,2009-2010,2026-03-18T10:29:47.580Z
502059,85206B,PINK FELT EASTER EGG BASKET,6,2010-03-22 14:44:00,1.65,16560,United Kingdom,2009-2010,2026-03-18T10:29:47.580Z
502059,85200,BUNNY EGG BOX,12,2010-03-22 14:44:00,1.25,16560,United Kingdom,2009-2010,2026-03-18T10:29:47.580Z
502059,22309,TEA COSY RED STRIPE,6,2010-03-22 14:44:00,2.55,16560,United Kingdom,2009-2010,2026-03-18T10:29:47.580Z
502059,21528,DAIRY MAID TRADITIONAL TEAPOT,2,2010-03-22 14:44:00,6.95,16560,United Kingdom,2009-2010,2026-03-18T10:29:47.580Z


## Cell 3 — Data Quality Assessment
Count nulls, negatives, and anomalies before cleaning

In [0]:
print("=" * 55)
print("  RAW DATA QUALITY ASSESSMENT")
print("=" * 55)

total = df_raw.count()

# Check actual column names first
print("Actual columns:", df_raw.columns)
print()

# Null counts per key column
for col in ["Invoice", "StockCode", "Description", "Quantity", "Price", "CustomerID", "Country"]:
    null_count = df_raw.filter(F.col(col).isNull() | (F.col(col) == "None") | (F.col(col) == "")).count()
    pct = round(null_count / total * 100, 1)
    flag = "⚠️ " if pct > 5 else "✅"
    print(f"  {flag} {col:<20} nulls: {null_count:>8,}  ({pct}%)")

# Negative quantities
neg_qty = df_raw.filter(F.col("Quantity").cast("double") < 0).count()
print(f"\n  ⚠️  Negative Quantity rows  : {neg_qty:,}  (returns/cancellations)")

# Zero price rows
zero_price = df_raw.filter(F.col("Price").cast("double") <= 0).count()
print(f"  ⚠️  Zero/negative Price rows: {zero_price:,}")

# Cancelled invoices
cancelled = df_raw.filter(F.col("Invoice").startswith("C")).count()
print(f"  ⚠️  Cancelled invoices (C*) : {cancelled:,}")

print("=" * 55)

  RAW DATA QUALITY ASSESSMENT
Actual columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'CustomerID', 'Country', 'source_sheet', 'ingested_at']

  ✅ Invoice              nulls:        0  (0.0%)
  ✅ StockCode            nulls:        0  (0.0%)
  ✅ Description          nulls:    4,382  (0.4%)
  ✅ Quantity             nulls:        0  (0.0%)
  ✅ Price                nulls:        0  (0.0%)
  ⚠️  CustomerID           nulls:  243,007  (22.8%)
  ✅ Country              nulls:        0  (0.0%)

  ⚠️  Negative Quantity rows  : 22,950  (returns/cancellations)
  ⚠️  Zero/negative Price rows: 6,207
  ⚠️  Cancelled invoices (C*) : 19,494


## Cell 4 — Clean & Type the POS Data (Bronze → Silver)
Apply all cleaning rules and produce silver_pos_cleaned

In [0]:
df_clean = df_raw

# ── STEP 1: Cast to correct data types ─────────────────────
df_clean = (df_clean
    .withColumn("invoice_no",      F.col("Invoice").cast(StringType()))
    .withColumn("stock_code",      F.col("StockCode").cast(StringType()))
    .withColumn("description",     F.trim(F.col("Description")))
    .withColumn("quantity",        F.col("Quantity").cast(IntegerType()))
    .withColumn("invoice_date",    F.to_timestamp("InvoiceDate"))
    .withColumn("unit_price_gbp",  F.col("Price").cast(DoubleType()))
    .withColumn("customer_id", F.col("CustomerID").cast(StringType()))
    .withColumn("country",         F.trim(F.col("Country")))
    .withColumn("source_sheet",    F.col("source_sheet"))
)

# ── STEP 2: Remove cancellations (Invoice starts with C) ───
df_clean = df_clean.filter(~F.col("invoice_no").startswith("C"))

# ── STEP 3: Remove rows with null/zero price or quantity ──
df_clean = df_clean.filter(
    F.col("quantity").isNotNull() &
    (F.col("quantity") > 0) &
    F.col("unit_price_gbp").isNotNull() &
    (F.col("unit_price_gbp") > 0)
)

# ── STEP 4: Remove rows with null StockCode ───────────────
df_clean = df_clean.filter(
    F.col("stock_code").isNotNull() &
    (F.col("stock_code") != "")
)

# ── STEP 5: Add derived date fields ───────────────────────
df_clean = (df_clean
    .withColumn("invoice_year",    F.year("invoice_date"))
    .withColumn("invoice_month",   F.month("invoice_date"))
    .withColumn("invoice_quarter", F.quarter("invoice_date"))
    .withColumn("invoice_week",    F.weekofyear("invoice_date"))
    .withColumn("month_label",     F.date_format("invoice_date", "yyyy-MM"))
)

# ── STEP 6: Calculate line revenue ────────────────────────
df_clean = df_clean.withColumn(
    "gross_revenue_gbp",
    F.round(F.col("quantity") * F.col("unit_price_gbp"), 2)
)

# ── STEP 7: Keep only UK transactions ─────────────────────
df_uk = df_clean.filter(F.col("country") == "United Kingdom")

print(f"Rows after cleaning (UK only): {df_uk.count():,}")
print(f"Removed non-UK, cancellations, nulls, negatives")

Rows after cleaning (UK only): 958,501
Removed non-UK, cancellations, nulls, negatives


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Deduplicate at invoice line level
window_dedup = Window.partitionBy("Invoice", "StockCode").orderBy(
    F.col("invoice_date").desc()
)

df_clean = df_clean.withColumn("row_num", F.row_number().over(window_dedup)) \
                   .filter(F.col("row_num") == 1) \
                   .drop("row_num")

print(f"Rows after deduplication: {df_clean.count():,}")

Rows after deduplication: 996,548


## Cell 5 — Assign Retailer & Channel
Map each transaction to a simulated retailer and channel (Retail vs DTC)

In [0]:
# Since the UCI dataset uses Customer IDs not retailer names,
# we simulate retailer assignment using customer ID ranges
# This mirrors how a real brand would segment their account base

df_assigned = df_uk.withColumn(
    "retailer_id",
    F.when(F.col("customer_id").isNull(),                          "RET-006")  # No ID = DTC
     .when(F.col("customer_id").cast("int") < 13000,               "RET-001")  # Boots
     .when(F.col("customer_id").cast("int").between(13000, 13999), "RET-002")  # John Lewis
     .when(F.col("customer_id").cast("int").between(14000, 14999), "RET-003")  # Asda
     .when(F.col("customer_id").cast("int").between(15000, 15499), "RET-004")  # Mothercare
     .when(F.col("customer_id").cast("int").between(15500, 15999), "RET-005")  # Smyths
     .when(F.col("customer_id").cast("int").between(16000, 16999), "RET-007")  # Amazon UK
     .when(F.col("customer_id").cast("int").between(17000, 17499), "RET-008")  # Very.co.uk
     .otherwise("RET-006")                                                      # DTC
)

## Cell 6 — Add UK Region
Join the region lookup to add UK regional dimension

In [0]:
df_regions = spark.table(f"{DATABASE}.silver_uk_region_lookup")

# Extract city from description or use a simplified mapping
# Since UCI data has no city field, we derive region from customer_id ranges
# This simulates how a real ERP would store delivery region

df_regional = df_assigned.withColumn(
    "uk_region",
    F.when(F.col("customer_id").cast("int") < 13000,               "London")
     .when(F.col("customer_id").cast("int").between(13000, 13499), "North West")
     .when(F.col("customer_id").cast("int").between(13500, 13999), "South East")
     .when(F.col("customer_id").cast("int").between(14000, 14499), "West Midlands")
     .when(F.col("customer_id").cast("int").between(14500, 14999), "Yorkshire")
     .when(F.col("customer_id").cast("int").between(15000, 15499), "Scotland")
     .when(F.col("customer_id").cast("int").between(15500, 15999), "East of England")
     .when(F.col("customer_id").cast("int").between(16000, 16499), "South West")
     .when(F.col("customer_id").cast("int").between(16500, 16999), "Wales")
     .when(F.col("customer_id").cast("int").between(17000, 17499), "North East")
     .otherwise("Other UK")
)

## Cell 7 — Join ERP Product Master & Retailer Funding

In [0]:
df_erp      = spark.table(f"{DATABASE}.silver_erp_product_master")
df_funding  = spark.table(f"{DATABASE}.silver_retailer_funding")

# Join retailer funding onto transactions
df_enriched = df_regional.join(
    df_funding.select(
        "retailer_id", "retailer_name", "channel",
        "rebate_pct", "settlement_disc_pct",
        "scan_allowance_per_unit", "fixed_annual_funding_gbp"
    ),
    on="retailer_id",
    how="left"
)

# Since UCI stock codes don't match our synthetic SKU IDs,
# we assign SKU categories based on stock_code hash
# This simulates a real ERP product master join

df_enriched = df_enriched.withColumn(
    "category",
    F.when(F.col("stock_code").rlike("^[0-9]{5}$"),
        F.element_at(
            F.array(
                F.lit("Feeding"), F.lit("Travel"), F.lit("Nursery"),
                F.lit("Sleep"),   F.lit("Bathing"), F.lit("Clothing"),
                F.lit("Toys"),    F.lit("Healthcare")
            ),
            (F.abs(F.hash(F.col("stock_code"))) % 8 + 1).cast("int")
        )
    ).otherwise("Other")
)

# Assign a cost price based on category average from ERP
category_costs = df_erp.groupBy("category").agg(
    F.avg("cost_price_gbp").alias("avg_cost"),
    F.avg("margin_pct").alias("avg_erp_margin_pct")
)

df_enriched = df_enriched.join(category_costs, on="category", how="left")

# Estimated cost per unit based on ERP category average cost-to-price ratio
df_enriched = df_enriched.withColumn(
    "cost_per_unit_gbp",
    F.round(
        F.col("unit_price_gbp") * (1 - F.coalesce(F.col("avg_erp_margin_pct"), F.lit(55)) / 100),
        2
    )
)

print(f"Enriched rows: {df_enriched.count():,}")

Enriched rows: 958,501


## Cell 8 — Build the P&L Waterfall
Gross Revenue → Net Revenue → Contribution Margin

In [0]:
df_pl = df_enriched

# ── P&L CALCULATIONS ───────────────────────────────────────

# 1. Gross Revenue
df_pl = df_pl.withColumn(
    "gross_revenue_gbp",
    F.round(F.col("quantity") * F.col("unit_price_gbp"), 2)
)

# 2. Rebate deduction (% of gross revenue)
df_pl = df_pl.withColumn(
    "rebate_deduction_gbp",
    F.round(F.col("gross_revenue_gbp") * F.coalesce(F.col("rebate_pct"), F.lit(0)) / 100, 2)
)

# 3. Settlement discount deduction
df_pl = df_pl.withColumn(
    "settlement_deduction_gbp",
    F.round(F.col("gross_revenue_gbp") * F.coalesce(F.col("settlement_disc_pct"), F.lit(0)) / 100, 2)
)

# 4. Scan allowance deduction (per unit)
df_pl = df_pl.withColumn(
    "scan_deduction_gbp",
    F.round(F.col("quantity") * F.coalesce(F.col("scan_allowance_per_unit"), F.lit(0)), 2)
)

# 5. Net Revenue = Gross - all deductions
df_pl = df_pl.withColumn(
    "net_revenue_gbp",
    F.round(
        F.col("gross_revenue_gbp")
        - F.col("rebate_deduction_gbp")
        - F.col("settlement_deduction_gbp")
        - F.col("scan_deduction_gbp"),
        2
    )
)

# 6. Cost of Goods Sold
df_pl = df_pl.withColumn(
    "cogs_gbp",
    F.round(F.col("quantity") * F.col("cost_per_unit_gbp"), 2)
)

# 7. Contribution Margin = Net Revenue - COGS
df_pl = df_pl.withColumn(
    "contribution_margin_gbp",
    F.round(F.col("net_revenue_gbp") - F.col("cogs_gbp"), 2)
)

# 8. Contribution Margin %
df_pl = df_pl.withColumn(
    "contribution_margin_pct",
    F.round(
        F.when(F.col("net_revenue_gbp") > 0,
            F.col("contribution_margin_gbp") / F.col("net_revenue_gbp") * 100
        ).otherwise(0),
        1
    )
)

# 9. Margin Leakage = Gross Revenue - Net Revenue
df_pl = df_pl.withColumn(
    "total_margin_leakage_gbp",
    F.round(F.col("gross_revenue_gbp") - F.col("net_revenue_gbp"), 2)
)

print("✅ P&L calculations complete")
display(df_pl.select(
    "invoice_no", "retailer_name", "channel", "category",
    "gross_revenue_gbp", "rebate_deduction_gbp",
    "settlement_deduction_gbp", "scan_deduction_gbp",
    "net_revenue_gbp", "cogs_gbp",
    "contribution_margin_gbp", "contribution_margin_pct"
).limit(10))

✅ P&L calculations complete


invoice_no,retailer_name,channel,category,gross_revenue_gbp,rebate_deduction_gbp,settlement_deduction_gbp,scan_deduction_gbp,net_revenue_gbp,cogs_gbp,contribution_margin_gbp,contribution_margin_pct
515189,DTC Ecommerce,DTC,Other,5.95,0.0,0.0,0.0,5.95,2.68,3.27,55.0
515189,DTC Ecommerce,DTC,Other,4.25,0.0,0.0,0.0,4.25,1.91,2.34,55.1
515192,DTC Ecommerce,DTC,Travel,52.2,0.0,0.0,0.0,52.2,18.36,33.84,64.8
515192,DTC Ecommerce,DTC,Sleep,15.0,0.0,0.0,0.0,15.0,4.72,10.28,68.5
515192,DTC Ecommerce,DTC,Toys,10.08,0.0,0.0,0.0,10.08,3.6,6.48,64.3
515192,DTC Ecommerce,DTC,Nursery,29.7,0.0,0.0,0.0,29.7,10.38,19.32,65.1
515192,DTC Ecommerce,DTC,Other,136.5,0.0,0.0,0.0,136.5,61.6,74.9,54.9
515192,DTC Ecommerce,DTC,Toys,97.5,0.0,0.0,0.0,97.5,34.0,63.5,65.1
515192,DTC Ecommerce,DTC,Sleep,63.5,0.0,0.0,0.0,63.5,20.1,43.4,68.3
515192,DTC Ecommerce,DTC,Toys,59.7,0.0,0.0,0.0,59.7,20.76,38.94,65.2


## Cell 9 — Write Silver POS Cleaned Table

In [0]:
# Select final Silver columns
silver_cols = [
    "invoice_no", "stock_code", "description", "category",
    "quantity", "invoice_date", "invoice_year", "invoice_month",
    "invoice_quarter", "invoice_week", "month_label",
    "unit_price_gbp", "cost_per_unit_gbp", "customer_id",
    "retailer_id", "retailer_name", "channel", "uk_region",
    "gross_revenue_gbp", "rebate_deduction_gbp",
    "settlement_deduction_gbp", "scan_deduction_gbp",
    "net_revenue_gbp", "cogs_gbp",
    "contribution_margin_gbp", "contribution_margin_pct",
    "total_margin_leakage_gbp", "source_sheet"
]

df_silver = df_pl.select(silver_cols)

(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("invoice_year", "invoice_month")
    .saveAsTable(f"{DATABASE}.silver_pos_cleaned"))

n = spark.sql(f"SELECT COUNT(*) AS n FROM {DATABASE}.silver_pos_cleaned").collect()[0]["n"]
print(f"✅ silver_pos_cleaned written: {n:,} rows")

✅ silver_pos_cleaned written: 958,501 rows


## Cell 10 — Gold: Retailer P&L Table

In [0]:
df_s = spark.table(f"{DATABASE}.silver_pos_cleaned")

gold_retailer_pl = df_s.groupBy(
    "retailer_id", "retailer_name", "channel",
    "invoice_year", "invoice_quarter", "month_label"
).agg(
    F.sum("gross_revenue_gbp").alias("gross_revenue_gbp"),
    F.sum("rebate_deduction_gbp").alias("rebate_deduction_gbp"),
    F.sum("settlement_deduction_gbp").alias("settlement_deduction_gbp"),
    F.sum("scan_deduction_gbp").alias("scan_deduction_gbp"),
    F.sum("total_margin_leakage_gbp").alias("total_leakage_gbp"),
    F.sum("net_revenue_gbp").alias("net_revenue_gbp"),
    F.sum("cogs_gbp").alias("cogs_gbp"),
    F.sum("contribution_margin_gbp").alias("contribution_margin_gbp"),
    F.avg("contribution_margin_pct").alias("avg_contribution_margin_pct"),
    F.sum("quantity").alias("total_units"),
    F.countDistinct("invoice_no").alias("total_invoices"),
    F.countDistinct("customer_id").alias("unique_customers")
).withColumn(
    "net_revenue_pct_of_gross",
    F.round(F.col("net_revenue_gbp") / F.col("gross_revenue_gbp") * 100, 1)
).withColumn(
    "margin_flag",
    F.when(F.col("avg_contribution_margin_pct") >= 40, "GREEN")
     .when(F.col("avg_contribution_margin_pct") >= 25, "AMBER")
     .otherwise("RED")
)

(gold_retailer_pl.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_retailer_pl"))

print(f"✅ gold_retailer_pl written: {gold_retailer_pl.count():,} rows")
display(gold_retailer_pl.orderBy("retailer_name", "month_label").limit(15))

✅ gold_retailer_pl written: 200 rows


retailer_id,retailer_name,channel,invoice_year,invoice_quarter,month_label,gross_revenue_gbp,rebate_deduction_gbp,settlement_deduction_gbp,scan_deduction_gbp,total_leakage_gbp,net_revenue_gbp,cogs_gbp,contribution_margin_gbp,avg_contribution_margin_pct,total_units,total_invoices,unique_customers,net_revenue_pct_of_gross,margin_flag
RET-007,Amazon UK,DTC,2009,4,2009-12,84729.05999999944,6780.26000000001,0.0,0.0,6780.26000000001,77948.80000000003,31275.26999999999,46673.529999999904,60.34389503361546,47488,211,149,92.0,GREEN
RET-007,Amazon UK,DTC,2010,1,2010-01,57023.01999999988,4562.509999999992,0.0,0.0,4562.509999999992,52460.50999999971,20865.12999999998,31595.38000000008,60.62685736988486,35513,122,96,92.0,GREEN
RET-007,Amazon UK,DTC,2010,1,2010-02,66916.39999999979,5354.619999999979,0.0,0.0,5354.619999999979,61561.77999999978,24302.640000000112,37259.14000000003,60.530347242486435,44115,156,122,92.0,GREEN
RET-007,Amazon UK,DTC,2010,1,2010-03,104371.30999999863,8351.690000000022,0.0,0.0,8351.690000000022,96019.62000000052,38204.31000000003,57815.31,60.76338416314525,58617,225,186,92.0,GREEN
RET-007,Amazon UK,DTC,2010,2,2010-04,89222.6799999991,7138.980000000016,0.0,0.0,7138.980000000016,82083.69999999995,32194.010000000155,49889.68999999996,61.25300880626259,56545,191,152,92.0,GREEN
RET-007,Amazon UK,DTC,2010,2,2010-05,71718.45999999977,5738.589999999991,0.0,0.0,5738.589999999991,65979.86999999992,25903.080000000053,40076.79000000014,61.32577957629165,44075,210,159,92.0,GREEN
RET-007,Amazon UK,DTC,2010,2,2010-06,106335.37999999903,8508.730000000001,0.0,0.0,8508.730000000001,97826.64999999978,38084.71000000021,59741.93999999994,61.14049631361253,71526,253,181,92.0,GREEN
RET-007,Amazon UK,DTC,2010,3,2010-07,76705.27999999926,6137.740000000039,0.0,0.0,6137.740000000039,70567.53999999978,27448.879999999925,43118.660000000025,61.339032855159864,46047,233,153,92.0,GREEN
RET-007,Amazon UK,DTC,2010,3,2010-08,88956.18999999961,7117.590000000009,0.0,0.0,7117.590000000009,81838.59999999969,31823.080000000045,50015.51999999982,61.46714553404153,58296,216,156,92.0,GREEN
RET-007,Amazon UK,DTC,2010,3,2010-09,105836.15999999842,8468.790000000005,0.0,0.0,8468.790000000005,97367.37000000036,37843.61000000014,59523.75999999981,61.34717680768551,65885,284,207,92.0,GREEN


## Cell 11 — Gold: SKU Profitability Ranking

In [0]:
gold_sku = df_s.groupBy(
    "stock_code", "description", "category"
).agg(
    F.sum("gross_revenue_gbp").alias("gross_revenue_gbp"),
    F.sum("net_revenue_gbp").alias("net_revenue_gbp"),
    F.sum("cogs_gbp").alias("cogs_gbp"),
    F.sum("contribution_margin_gbp").alias("contribution_margin_gbp"),
    F.avg("contribution_margin_pct").alias("avg_contribution_margin_pct"),
    F.sum("quantity").alias("total_units_sold"),
    F.avg("unit_price_gbp").alias("avg_selling_price_gbp"),
    F.avg("cost_per_unit_gbp").alias("avg_cost_per_unit_gbp"),
    F.countDistinct("retailer_id").alias("num_retailers_stocking")
)

# Add SKU performance flag
gold_sku = gold_sku.withColumn(
    "sku_flag",
    F.when(F.col("avg_contribution_margin_pct") >= 45, "HERO")
     .when(F.col("avg_contribution_margin_pct") >= 30, "SOLID")
     .when(F.col("avg_contribution_margin_pct") >= 15, "WATCH")
     .otherwise("AT RISK")
)

# Revenue rank
window_rev = Window.orderBy(F.col("gross_revenue_gbp").desc())
gold_sku = gold_sku.withColumn("revenue_rank", F.rank().over(window_rev))

# Margin rank
window_mgn = Window.orderBy(F.col("avg_contribution_margin_pct").desc())
gold_sku = gold_sku.withColumn("margin_rank", F.rank().over(window_mgn))

(gold_sku.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_sku_profitability"))

print(f"✅ gold_sku_profitability written: {gold_sku.count():,} SKUs")

# Show top 15 by revenue
display(gold_sku.orderBy("revenue_rank").limit(15))

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ gold_sku_profitability written: 5,571 SKUs


stock_code,description,category,gross_revenue_gbp,net_revenue_gbp,cogs_gbp,contribution_margin_gbp,avg_contribution_margin_pct,total_units_sold,avg_selling_price_gbp,avg_cost_per_unit_gbp,num_retailers_stocking,sku_flag,revenue_rank,margin_rank
DOT,DOTCOM POSTAGE,Other,322657.4800000001,321756.5000000001,145196.14999999997,176560.35,54.953969359331474,1436,224.6918384401115,101.11152506963786,2,HERO,1,2836
22423,REGENCY CAKESTAND 3 TIER,Sleep,291249.3499999999,275388.41,92035.60999999999,183352.79999999993,66.20861779518742,22911,14.433942361499717,4.5615081141577996,8,HERO,2,114
M,Manual,Other,245954.48000000004,240646.39000000004,110665.04,129981.34999999998,29.07814910025707,9949,308.67273778920304,138.90271208226218,8,WATCH,3,4925
85123A,WHITE HANGING HEART T-LIGHT HOLDER,Other,242731.75999999992,208303.17,109433.50000000003,98869.67000000001,47.763862472567624,88410,3.0827121433796623,1.3896817849305063,8,HERO,4,3991
23843,"PAPER CRAFT , LITTLE BIRDIE",Feeding,168469.6,154992.03,52646.75,102345.28,66.0,80995,2.08,0.65,1,HERO,5,145
47566,PARTY BUNTING,Feeding,140266.70000000007,130647.91000000003,43847.549999999996,86800.36000000002,65.808506543495,26317,5.778487297921479,1.806451116243264,8,HERO,6,165
85099B,JUMBO BAG RED RETROSPOT,Other,136487.61000000004,111753.98,61503.080000000016,50250.90000000001,46.09353091140854,72082,2.395656469088591,1.0802166985340986,8,HERO,7,4165
84879,ASSORTED COLOUR BIRD ORNAMENT,Clothing,123045.47999999995,96332.26000000001,45096.55,51235.70999999999,52.878509719222485,76013,1.8667494600431953,0.6853131749460039,8,HERO,8,3249
22086,PAPER CHAIN KIT 50'S CHRISTMAS,Feeding,120037.49000000002,109479.48000000004,37540.60000000003,71938.88000000006,64.32676649508657,35442,3.372035563874589,1.0528638277959756,8,HERO,9,432
79321,CHILLI LIGHTS,Sleep,84382.01000000002,77927.71000000002,26629.520000000004,51298.189999999995,65.68465517241378,16622,6.288801724137931,1.9858275862068966,8,HERO,10,179


## Cell 12 — Gold: Regional Performance Table

In [0]:
gold_regional = df_s.groupBy(
    "uk_region", "invoice_year", "invoice_quarter", "month_label"
).agg(
    F.sum("gross_revenue_gbp").alias("gross_revenue_gbp"),
    F.sum("net_revenue_gbp").alias("net_revenue_gbp"),
    F.sum("contribution_margin_gbp").alias("contribution_margin_gbp"),
    F.avg("contribution_margin_pct").alias("avg_contribution_margin_pct"),
    F.sum("quantity").alias("total_units"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.countDistinct("invoice_no").alias("total_orders")
).withColumn(
    "region_flag",
    F.when(F.col("avg_contribution_margin_pct") >= 40, "GREEN")
     .when(F.col("avg_contribution_margin_pct") >= 25, "AMBER")
     .otherwise("RED")
)

(gold_regional.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_regional_performance"))

print(f"✅ gold_regional_performance written: {gold_regional.count():,} rows")
display(gold_regional.orderBy(F.col("gross_revenue_gbp").desc()).limit(15))

✅ gold_regional_performance written: 275 rows


uk_region,invoice_year,invoice_quarter,month_label,gross_revenue_gbp,net_revenue_gbp,contribution_margin_gbp,avg_contribution_margin_pct,total_units,unique_customers,total_orders,region_flag
Other UK,2010,4,2010-12,549901.8199999784,549901.8199999784,347652.60000000556,64.13392525841634,145153,138,405,GREEN
Other UK,2011,4,2011-11,484011.5200000361,484011.5200000361,311613.3900000042,64.71773482659154,177481,219,440,GREEN
Other UK,2010,4,2010-11,431213.7399999868,431213.7399999868,276923.56000000087,64.20727867511845,140000,225,507,GREEN
Other UK,2009,4,2009-12,264851.34000000294,264851.34000000294,164774.7000000022,63.309830849105836,107472,130,411,GREEN
Other UK,2011,4,2011-10,260622.0100000069,260622.0100000069,167869.92000000234,64.53162110927713,98181,161,350,GREEN
Other UK,2011,3,2011-09,249525.01000000458,249525.01000000458,160192.70000000106,64.4519326203964,98419,155,313,GREEN
Other UK,2010,1,2010-03,240250.82000000498,240250.82000000498,150358.710000003,63.55646330536719,120928,138,371,GREEN
Other UK,2010,4,2010-10,235631.37000000713,235631.37000000713,147941.58000000272,64.12292092438236,70835,159,398,GREEN
Other UK,2011,2,2011-06,214587.43000000375,214587.43000000375,138134.77000000022,64.33680294423652,76400,126,326,GREEN
Other UK,2011,3,2011-07,214153.62000000582,214153.62000000582,136193.27000000034,64.15214976599191,81334,132,340,GREEN


## Cell 13 — Gold: Margin Waterfall Bridge Table
Used for the waterfall chart in Power BI

In [0]:
# Aggregate total P&L components across all retailers
waterfall_data = df_s.agg(
    F.sum("gross_revenue_gbp").alias("gross_revenue"),
    F.sum("rebate_deduction_gbp").alias("rebate_deductions"),
    F.sum("settlement_deduction_gbp").alias("settlement_deductions"),
    F.sum("scan_deduction_gbp").alias("scan_allowances"),
    F.sum("net_revenue_gbp").alias("net_revenue"),
    F.sum("cogs_gbp").alias("cogs"),
    F.sum("contribution_margin_gbp").alias("contribution_margin")
).collect()[0]

# Build waterfall rows
import pandas as pd

waterfall_rows = [
    ("1_Gross_Revenue",        "Gross Revenue",         float(waterfall_data["gross_revenue"]),      "base"),
    ("2_Rebates",              "Rebate Deductions",     -float(waterfall_data["rebate_deductions"]),  "negative"),
    ("3_Settlement",           "Settlement Discounts",  -float(waterfall_data["settlement_deductions"]), "negative"),
    ("4_Scan_Allowances",      "Scan Allowances",       -float(waterfall_data["scan_allowances"]),    "negative"),
    ("5_Net_Revenue",          "Net Revenue",            float(waterfall_data["net_revenue"]),         "subtotal"),
    ("6_COGS",                 "Cost of Goods Sold",    -float(waterfall_data["cogs"]),               "negative"),
    ("7_Contribution_Margin",  "Contribution Margin",    float(waterfall_data["contribution_margin"]), "total"),
]

df_waterfall_pd = pd.DataFrame(waterfall_rows, columns=["sort_key","pl_line","amount_gbp","bar_type"])
df_waterfall_pd["amount_gbp"] = df_waterfall_pd["amount_gbp"].round(2)

df_waterfall = spark.createDataFrame(df_waterfall_pd)

(df_waterfall.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_margin_waterfall"))

print("✅ gold_margin_waterfall written")
display(df_waterfall)

✅ gold_margin_waterfall written


sort_key,pl_line,amount_gbp,bar_type
1_Gross_Revenue,Gross Revenue,1.787097776E7,base
2_Rebates,Rebate Deductions,-610108.52,negative
3_Settlement,Settlement Discounts,-161478.38,negative
4_Scan_Allowances,Scan Allowances,-2080194.25,negative
5_Net_Revenue,Net Revenue,1.501919661E7,subtotal
6_COGS,Cost of Goods Sold,-6418703.04,negative
7_Contribution_Margin,Contribution Margin,8600493.57,total


## Cell 14 — Module 1 Summary & Quality Check

In [0]:
print("=" * 65)
print("  MODULE 1 — QUALITY CHECK & COMMERCIAL SUMMARY")
print("=" * 65)

tables = [
    "silver_pos_cleaned",
    "gold_retailer_pl",
    "gold_sku_profitability",
    "gold_regional_performance",
    "gold_margin_waterfall",
]

for t in tables:
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {DATABASE}.{t}").collect()[0]["n"]
    print(f"  ✅  {t:<35} {n:>10,} rows")

print("=" * 65)

# Commercial headline numbers
df_s = spark.table(f"{DATABASE}.silver_pos_cleaned")
summary = df_s.agg(
    F.round(F.sum("gross_revenue_gbp"), 0).alias("Total Gross Revenue GBP"),
    F.round(F.sum("net_revenue_gbp"), 0).alias("Total Net Revenue GBP"),
    F.round(F.sum("contribution_margin_gbp"), 0).alias("Total Contribution Margin GBP"),
    F.round(F.avg("contribution_margin_pct"), 1).alias("Avg Contribution Margin Pct"),
    F.round(F.sum("total_margin_leakage_gbp"), 0).alias("Total Margin Leakage GBP"),
    F.countDistinct("invoice_no").alias("Total Invoices"),
    F.countDistinct("customer_id").alias("Unique Customers"),
    F.countDistinct("stock_code").alias("Unique SKUs"),
).collect()[0]

print()
print("  💰 COMMERCIAL HEADLINE NUMBERS")
print("  " + "-" * 45)
for key in summary.asDict():
    val = summary[key]
    if isinstance(val, float):
        print(f"  {key:<35} {val:>12,.1f}")
    else:
        print(f"  {key:<35} {val:>12,}")

print("=" * 65)
print("  🎉 MODULE 1 COMPLETE!")
print("  👉 Next: Module 2 — Trade Spend & Promotional ROI")

  MODULE 1 — QUALITY CHECK & COMMERCIAL SUMMARY
  ✅  silver_pos_cleaned                     958,501 rows
  ✅  gold_retailer_pl                           200 rows
  ✅  gold_sku_profitability                   5,571 rows
  ✅  gold_regional_performance                  275 rows
  ✅  gold_margin_waterfall                        7 rows

  💰 COMMERCIAL HEADLINE NUMBERS
  ---------------------------------------------
  Total Gross Revenue GBP             17,870,978.0
  Total Net Revenue GBP               15,019,197.0
  Total Contribution Margin GBP        8,600,494.0
  Avg Contribution Margin Pct                 45.0
  Total Margin Leakage GBP             2,851,781.0
  Total Invoices                            36,535
  Unique Customers                           5,350
  Unique SKUs                                4,906
  🎉 MODULE 1 COMPLETE!
  👉 Next: Module 2 — Trade Spend & Promotional ROI
